# 06 - Clustering de Perfiles de Carga (DTW)

**Módulo 3 - Series de Tiempo | ML Avanzado**

Cerramos el módulo con aprendizaje **no supervisado**: agrupar *series
completas* por la **forma** de su trayectoria. Con nuestro dataset horario,
cada **día** del hogar es una curva de 24 puntos — su *perfil de carga*.
¿Existen arquetipos (día laboral, fin de semana, vacaciones)? Es exactamente
lo que hacen las eléctricas para segmentar clientes.

La herramienta central: **clustering con DTW** (*Dynamic Time Warping*) —
agrupar días por la *forma* de su curva, tolerando rutinas desfasadas en el
tiempo.

Al ser no supervisado, aquí no hay pronóstico que evaluar: en lugar de las
métricas de error usamos el **Índice de Rand Ajustado** contra etiquetas
conocidas (¿laboral o fin de semana?).

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

# Hacemos importable utils/ tanto si el notebook corre desde notebooks/ como
# desde la raíz del repositorio.
_here = os.getcwd()
for cand in (os.path.join(_here, "..", "utils"), os.path.join(_here, "utils"),
             os.path.join(_here, "..", "..", "module3-time-series", "utils")):
    cand = os.path.abspath(cand)
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow_helpers import setup_mlflow, log_and_register, register_best_run

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
np.random.seed(42)
print("Versión de MLflow:", mlflow.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Carga del dataset UCI #235 (con caché local y respaldo sintético offline)
# ---------------------------------------------------------------------------
import io, zipfile, urllib.request

UCI_ZIP_URL = ("https://archive.ics.uci.edu/static/public/235/"
               "individual+household+electric+power+consumption.zip")

def _data_dir():
    for cand in ("../data", "data", "module3-time-series/data"):
        cand = os.path.abspath(cand)
        if os.path.isdir(cand):
            return cand
    cand = os.path.abspath("../data")
    os.makedirs(cand, exist_ok=True)
    return cand

DATA_DIR = _data_dir()
DAILY_CSV = os.path.join(DATA_DIR, "household_power_daily.csv")
HOURLY_CSV = os.path.join(DATA_DIR, "household_power_hourly.csv")

def load_household_power():
    """Devuelve (daily, hourly): potencia activa global media, en kW."""
    if os.path.isfile(DAILY_CSV) and os.path.isfile(HOURLY_CSV):
        daily = pd.read_csv(DAILY_CSV, index_col=0, parse_dates=True).iloc[:, 0]
        hourly = pd.read_csv(HOURLY_CSV, index_col=0, parse_dates=True).iloc[:, 0]
        return daily.asfreq("D"), hourly.asfreq("h")

    print("Descargando el dataset UCI #235 (~20 MB)...")
    raw = urllib.request.urlopen(UCI_ZIP_URL, timeout=180).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        with zf.open("household_power_consumption.txt") as fh:
            df = pd.read_csv(fh, sep=";", na_values=["?"], low_memory=False,
                             usecols=["Date", "Time", "Global_active_power"])
    ts = pd.to_datetime(df["Date"] + " " + df["Time"],
                        format="%d/%m/%Y %H:%M:%S")
    power = pd.Series(df["Global_active_power"].astype(float).to_numpy(),
                      index=ts, name="global_active_power_kw").sort_index()

    # Agregamos y rellenamos huecos por interpolación temporal (~1.25% de
    # minutos faltantes + un par de cortes de varios días).
    daily = power.resample("D").mean().interpolate(method="time")
    hourly = power.resample("h").mean().interpolate(method="time")
    daily = daily.iloc[1:-1]                       # primer/último día parciales
    hourly = hourly.loc[daily.index.min():
                        daily.index.max() + pd.Timedelta(hours=23)]
    daily.to_frame().to_csv(DAILY_CSV)
    hourly.to_frame().to_csv(HOURLY_CSV)
    return daily.asfreq("D"), hourly.asfreq("h")

try:
    daily, hourly = load_household_power()
    print(f"daily : {daily.index.min().date()} .. {daily.index.max().date()} "
          f"(n={len(daily)})")
    print(f"hourly: n={len(hourly)}")
except Exception as e:
    print("No se pudo descargar el dataset:", repr(e))
    print("Usando RESPALDO SINTÉTICO (estacionalidad semanal + anual).")
    rng = np.random.default_rng(7)
    idx = pd.date_range("2006-12-17", "2010-11-25", freq="D")
    t = np.arange(len(idx))
    daily = pd.Series(
        1.1
        + 0.35 * np.cos(2 * np.pi * (t - 20) / 365.25)   # invierno alto
        + 0.10 * (idx.dayofweek >= 5)                     # fin de semana
        + rng.normal(0, 0.12, len(idx)),
        index=idx, name="global_active_power_kw").clip(lower=0.1).asfreq("D")
    hidx = pd.date_range(idx.min(), idx.max() + pd.Timedelta(hours=23), freq="h")
    hh = hidx.hour.to_numpy()
    base = daily.reindex(pd.DatetimeIndex(hidx.date)).to_numpy()
    profile = 0.6 + 0.35 * np.sin(2 * np.pi * (hh - 14) / 24) \
              + 0.25 * ((hh >= 18) & (hh <= 22))
    hourly = pd.Series(base * profile + rng.normal(0, 0.05, len(hidx)),
                       index=hidx, name=daily.name).clip(lower=0.05).asfreq("h")

## 1. De la serie horaria a una matriz de perfiles diarios

Pivotamos: filas = días, columnas = las 24 horas. Para que el costo $O(n^2)$
de DTW sea manejable, muestreamos 200 días al azar.

In [ ]:
prof = hourly.to_frame("kw")
prof["date"] = prof.index.normalize()
prof["hour"] = prof.index.hour
mat = prof.pivot_table(index="date", columns="hour", values="kw").dropna()
print("matriz de perfiles:", mat.shape)

rng = np.random.default_rng(0)
sample_days = np.sort(rng.choice(len(mat), size=min(200, len(mat)),
                                 replace=False))
mat_s = mat.iloc[sample_days]
is_weekend = (pd.DatetimeIndex(mat_s.index).dayofweek >= 5).astype(int)
print(f"muestra: {len(mat_s)} días ({is_weekend.sum()} de fin de semana)")

fig, ax = plt.subplots(figsize=(11, 4))
for row in mat_s.to_numpy()[:60]:
    ax.plot(row, color="0.6", alpha=0.35)
ax.plot(mat_s.mean(axis=0), color="C3", lw=2.5, label="perfil medio")
ax.set_xlabel("hora del día"); ax.set_ylabel("kW")
ax.set_title("Perfiles de carga diarios (60 días de muestra)")
ax.legend(); plt.tight_layout(); plt.show()

## 2. z-normalización: agrupar por forma, no por nivel

Estandarizamos cada perfil a media 0 y varianza 1:

$$
z_t = \frac{y_t - \mu}{\sigma} .
$$

Sin esto, la distancia queda dominada por el **nivel** (invierno vs verano) y
no por el **patrón horario** (madrugar vs trasnochar). Sáltatela solo si el
nivel absoluto es justamente lo que quieres agrupar.

In [ ]:
def znorm(arr):
    mu = arr.mean(axis=1, keepdims=True)
    sd = arr.std(axis=1, keepdims=True)
    sd[sd == 0] = 1.0
    return (arr - mu) / sd

dataz = znorm(mat_s.to_numpy())

## 3. ¿Por qué una distancia especial? Euclidiana vs DTW

La distancia **Euclidiana** compara punto a punto:
$d_{euc}(a,b) = \sqrt{\sum_i (a_i - b_i)^2}$ — es **rígida en el tiempo**:
dos días con la misma rutina pero desplazada una hora (¡cenar a las 20 vs a
las 21!) parecen muy distintos.

**Dynamic Time Warping (DTW)** busca el mejor **alineamiento no lineal**
entre los dos ejes de tiempo. Con costo local $c(i,j) = (a_i - b_j)^2$ y la
recurrencia de programación dinámica:

$$
D(i,j) = c(i,j) + \min\{\, D(i-1,j),\; D(i,j-1),\; D(i-1,j-1) \,\},
$$

la distancia es $\sqrt{D(n,p)}$. El **camino de deformación** que traza los
$\min$ debe ser de frontera (de $(1,1)$ a $(n,p)$), monótono y continuo.
Restricciones tipo **banda de Sakoe-Chiba** limitan cuánto se aleja de la
diagonal (acelera y evita deformaciones patológicas). Costo: $O(np)$ por
par.

In [ ]:
# DTW diminuto y sin dependencias, para que la idea sea concreta.
def dtw_distance(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    n, p = len(a), len(b)
    D = np.full((n + 1, p + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, p + 1):
            cost = (a[i - 1] - b[j - 1]) ** 2
            D[i, j] = cost + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return np.sqrt(D[n, p])

# Demo con perfiles reales: un día vs el mismo perfil desplazado 2 horas
a = dataz[0]
b = np.roll(a, 2)
print("Euclidiana:", round(float(np.sqrt(np.sum((a - b) ** 2))), 3))
print("DTW       :", round(float(dtw_distance(a, b)), 3),
      " (menor -> reconoce la misma rutina desfasada)")

## 4a. `tslearn`: TimeSeriesKMeans con DTW + centroides DBA

k-means bajo DTW con centroides **DBA** (*DTW Barycenter Averaging*): una
"forma promedio" coherente con DTW, en vez de una media punto a punto.
Empezamos con $k=2$ — la hipótesis natural: ¿laboral vs fin de semana?
(Si `tslearn` no está instalado, seguimos con la vía scipy de la sección
4b.)

In [ ]:
km_labels = None
try:
    from tslearn.clustering import TimeSeriesKMeans
    from tslearn.utils import to_time_series_dataset

    K = 2
    km = TimeSeriesKMeans(n_clusters=K, metric="dtw", max_iter=10,
                          random_state=0)
    km_labels = km.fit_predict(to_time_series_dataset(dataz))

    fig, axes = plt.subplots(1, K, figsize=(12, 3.6), sharey=True)
    for c, ax in enumerate(np.atleast_1d(axes)):
        for row in dataz[km_labels == c]:
            ax.plot(row, color="0.7", alpha=0.35)
        ax.plot(km.cluster_centers_[c].ravel(), color="C3", lw=2.5,
                label="centroide DBA")
        ax.set_title(f"cluster {c} (n={int(np.sum(km_labels == c))})")
        ax.set_xlabel("hora"); ax.legend()
    fig_clusters = fig
    plt.tight_layout(); plt.show()
except Exception as e:
    print("tslearn no disponible - usaremos el jerárquico de scipy:", repr(e))

## 4b. Jerárquico de scipy sobre la matriz DTW

Alternativa con pocas dependencias: matriz completa de distancias DTW por
pares + clustering **aglomerativo** con enlace promedio. Ventaja: el
**dendrograma** deja elegir $k$ *después* de ver la estructura.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

N = dataz.shape[0]
dist = np.zeros((N, N))
for i in range(N):
    for j in range(i + 1, N):
        d = dtw_distance(dataz[i], dataz[j])
        dist[i, j] = dist[j, i] = d
print("matriz DTW:", dist.shape)

Z = linkage(squareform(dist, checks=False), method="average")

fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(Z, ax=ax, no_labels=True,
           color_threshold=0.7 * np.max(Z[:, 2]))
ax.set_title("Dendrograma - jerárquico (enlace promedio) sobre DTW")
ax.set_ylabel("distancia de enlace")
plt.tight_layout(); plt.show()

hier_labels = fcluster(Z, t=2, criterion="maxclust")
print("tamaños de clusters:", np.bincount(hier_labels)[1:])

## 5. Interpretación y validación con ARI

Los IDs de cluster son arbitrarios; el **Índice de Rand Ajustado** (1.0 =
partición idéntica, ≈0 = azar) es invariante al reetiquetado. Contrastamos
contra la etiqueta *fin de semana* y cruzamos con la tabla de contingencia:
¿los clusters descubren la rutina laboral?

In [ ]:
from sklearn.metrics import adjusted_rand_score

ari_km = (adjusted_rand_score(is_weekend, km_labels)
          if km_labels is not None else float("nan"))
ari_h = adjusted_rand_score(is_weekend, hier_labels)
if km_labels is not None:
    print("ARI k-means DTW vs fin de semana :", round(ari_km, 3))
print("ARI jerárquico vs fin de semana  :", round(ari_h, 3))

labels_show = km_labels if km_labels is not None else hier_labels
ct = pd.crosstab(pd.Series(labels_show, name="cluster"),
                 pd.Series(np.where(is_weekend == 1, "finde", "laboral"),
                           name="tipo de día"))
display(ct)

# Un ARI cercano a 0 dice que la partición NO coincide con laboral/finde:
# tras z-normalizar y permitir deformación temporal (DTW), la FORMA del día
# laboral y la del finde de este hogar son muy parecidas — la deformación
# absorbe justo los desfases (levantarse más tarde) que los distinguían.
# Lo que sí emerge es un grupo pequeño de días atípicos (vacaciones /
# ausencias, perfiles planos). El ARI valida HIPÓTESIS: aquí rechaza la
# nuestra, y un resultado negativo también es un hallazgo. Sube k (3, 4...)
# o agrupa SIN z-normalizar (nivel invierno/verano) y compara.

## 6. Registro en MLflow

In [ ]:
setup_mlflow("module3-06-clustering", backend="dagshub")

figs = {}
if km_labels is not None:
    figs["plots/clusters.png"] = fig_clusters
log_and_register(
    run_name="dtw-load-profiles",
    params={"n_days": int(N), "k": 2, "metric": "dtw",
            "znorm": True, "dataset": "uci-household-power"},
    metrics={**({"ARI_kmeans_weekend": float(ari_km)}
               if km_labels is not None else {}),
             "ARI_hierarchical_weekend": float(ari_h)},
    tags={"notebook": "06_timeseries_clustering", "familia": "unsupervised"},
    figures=figs,
)

## Resumen

- Cada día del hogar es un **perfil de carga** de 24 puntos; agruparlos revela
  arquetipos de comportamiento.
- **z-normaliza** para agrupar por *forma* y no por nivel; usa **DTW** (no
  Euclidiana) para tolerar rutinas desfasadas — recurrencia
  $D(i,j) = c(i,j) + \min\{\dots\}$ con camino de deformación.
- Herramientas: `tslearn` **TimeSeriesKMeans (DTW + DBA)** o el **jerárquico
  de scipy** sobre la matriz DTW (el dendrograma ayuda a elegir $k$).
- Valida contra etiquetas conocidas con el **Índice de Rand Ajustado** — y
  acepta el veredicto: aquí el ARI ≈ 0 **rechaza** la hipótesis laboral/finde
  (DTW + z-norm hacen muy parecidas ambas rutinas) y lo que emerge es el
  arquetipo *día atípico / vacaciones*. Un resultado negativo bien validado
  también es un hallazgo.

**Fin del Módulo 3.** El recorrido completo: descomposición y baselines →
suavizamiento exponencial y SARIMA → ingeniería de variables → XGBoost →
ensambles → clustering, todo sobre el mismo dataset real y versionado en
MLflow.